# **T-learner**

The T-learner (Two learner) solves the S-learner's biggest weakness; where tree-based models can accidentally ignore the treatment variable ($X$) during training because powerful confounders ($Z$) overshadow it. By training two completely independent models, the T-learner forces the algorithm to account for the treatment.

## **The Churn Scenario Setup**

Imagine you run a subscription software company and want to know: Does offering a $20 discount code ($X$) actually prevent customers from churning ($Y$)?

In your historical data, you have:
- Treatment ($X$): $1$ if the customer received a discount, $0$ if they did not.
- Outcome ($Y$): $1$ if the customer churned (canceled), $0$ if they stayed.- Confounders ($Z$): Variables that affect both whether a customer gets a discount and whether they churn.
    - Tenure (months subscribed)
    - Weekly app logins
    - Support tickets
    - Monthly spend tier

## **Implementation**

### **Training and Testing of the Model**

In [10]:
# Import necessary libraries
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split,GridSearchCV, StratifiedKFold
from sklearn.metrics import  recall_score, precision_score, f1_score,roc_auc_score

**Generate synthetic data for demonstration**

In [11]:
def generate_enterprise_churn_data(n=10000, seed=42):
    np.random.seed(seed)
    
    # 1. Realistic Confounders
    # Tenure: Gamma distribution (skewed: most customers are newer, fewer are multi-year veterans)
    tenure_months = (np.clip(np.random.gamma(shape=2, scale=6, size=n), 1, 60)).astype(int)
    
    # Weekly Logins: Poisson distribution
    weekly_logins = np.random.poisson(lam=4, size=n)
    
    # Support Tickets: Higher tickets usually indicate friction
    support_tickets = np.random.poisson(lam=1.5, size=n)
    
    # Monthly Spend Tier ($10, $25, $50, $100)
    monthly_spend = np.random.choice([10, 25, 50, 100], size=n, p=[0.4, 0.3, 0.2, 0.1])
    
    # 2. Confounding Mechanism (Propensity Score)
    # Retention teams historically give discounts to customers with low tenure, low logins, AND high support tickets.
    propensity_logit = -0.5 + (0.05 * tenure_months) - (0.20 * weekly_logins) + (0.40 * support_tickets) - (0.005 * monthly_spend)
    discount_prob = 1 / (1 + np.exp(-propensity_logit))
    treatment_x = np.random.binomial(1, discount_prob)
    
    # 3. Outcome Mechanism (Churn)
    # True Causal Effect: Offering a discount reduces the log-odds of churning by -0.35 (~15-20% absolute drop)
    true_causal_effect = -0.35
    
    baseline_logit = (
        1.2 
        - (0.06 * tenure_months) 
        - (0.15 * weekly_logins) 
        + (0.50 * support_tickets) 
        - (0.01 * monthly_spend)
        + (true_causal_effect * treatment_x) # The true causal impact!
    )
    churn_prob = 1 / (1 + np.exp(-baseline_logit))
    outcome_y = np.random.binomial(1, churn_prob)
    
    df = pd.DataFrame({
        'tenure_months': tenure_months,
        'weekly_logins': weekly_logins,
        'support_tickets': support_tickets,
        'monthly_spend': monthly_spend,
        'discount_given': treatment_x, # Treatment (X)
        'churned': outcome_y           # Outcome (Y)
    })
    
    return df

In [12]:
df = generate_enterprise_churn_data(n=10000)

In [13]:
df.head()

,tenure_months,weekly_logins,support_tickets,monthly_spend,discount_given,churned
0,14,7,1,25,0,0
1,8,4,3,50,0,0
2,8,9,2,10,1,1
3,8,7,0,10,0,0
4,27,3,1,100,0,1


**Split the data**

In [14]:
# train test split
X = df.drop(columns=['churned'])
y = df['churned']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=50)

In [15]:
print(f"Dataset Split Sizes:")
print(f" - Train Set: {len(X_train)} rows")
print(f" - Test Set: {len(X_test)} rows\n")

Dataset Split Sizes:
 - Train Set: 8000 rows
 - Test Set: 2000 rows



**Prepare Subsets for T-Learner**

In [16]:
# Note: In a T-learner, 'discount_given' is EXCLUDED from the feature set 
# because the treatment is split into two separate models entirely.
features = ['tenure_months', 'weekly_logins', 'support_tickets', 'monthly_spend']

In [17]:
# Split the training data into two groups based on treatment
train_mask_0 = (X_train['discount_given'] == 0)
train_mask_1 = (X_train['discount_given'] == 1)

X_train_0 = X_train.loc[train_mask_0, features]
y_train_0 = y_train[train_mask_0]

X_train_1 = X_train.loc[train_mask_1, features]
y_train_1 = y_train[train_mask_1]

In [18]:
print(f"Training Subsets:")
print(f" - Control Group (X=0):   {len(X_train_0)} rows")
print(f" - Treatment Group (X=1): {len(X_train_1)} rows\n")

Training Subsets:
 - Control Group (X=0):   4502 rows
 - Treatment Group (X=1): 3498 rows



**Train Model f0 (Control Group: No Discount)**

In [19]:
rf_0 = RandomForestClassifier(random_state=50)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=50)
param_grid = {
    'n_estimators': [50, 150, 250],
    'max_depth': [10, 15, 20]
}
scoring_metrics = {'AUC': 'roc_auc', 'R': 'recall', 'P': 'precision', 'F1': 'f1'}

grid_0 = GridSearchCV(
    estimator=rf_0, 
    param_grid=param_grid, 
    scoring=scoring_metrics,
    cv=cv, refit="AUC", 
    n_jobs=-2, 
    verbose=3
)

In [20]:
print("Starting Grid Search for Model f0 (Control)...")
grid_0.fit(X_train_0, y_train_0)
best_model_0 = grid_0.best_estimator_
print(f"Model f0 Best CV ROC AUC: {np.round(grid_0.best_score_, 2)}")

Starting Grid Search for Model f0 (Control)...
Fitting 5 folds for each of 9 candidates, totalling 45 fits
[CV 1/5] END max_depth=10, n_estimators=50; AUC: (test=0.629) F1: (test=0.650) P: (test=0.639) R: (test=0.662) total time=   0.9s
[CV 2/5] END max_depth=10, n_estimators=50; AUC: (test=0.679) F1: (test=0.694) P: (test=0.661) R: (test=0.731) total time=   0.4s
[CV 3/5] END max_depth=10, n_estimators=50; AUC: (test=0.672) F1: (test=0.684) P: (test=0.651) R: (test=0.720) total time=   0.3s
[CV 4/5] END max_depth=10, n_estimators=50; AUC: (test=0.676) F1: (test=0.693) P: (test=0.647) R: (test=0.746) total time=   0.3s
[CV 5/5] END max_depth=10, n_estimators=50; AUC: (test=0.652) F1: (test=0.666) P: (test=0.629) R: (test=0.708) total time=   0.3s
[CV 1/5] END max_depth=10, n_estimators=150; AUC: (test=0.629) F1: (test=0.643) P: (test=0.633) R: (test=0.654) total time=   2.6s
[CV 2/5] END max_depth=10, n_estimators=150; AUC: (test=0.677) F1: (test=0.680) P: (test=0.642) R: (test=0.723) 

**Train Model f1 (Treatment Group: Discount)**

In [21]:
rf_1 = RandomForestClassifier(random_state=50)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=50)
param_grid = {
    'n_estimators': [50, 150, 250],
    'max_depth': [10, 15, 20]
}
scoring_metrics = {'AUC': 'roc_auc', 'R': 'recall', 'P': 'precision', 'F1': 'f1'}

grid_1 = GridSearchCV(
    estimator=rf_1, 
    param_grid=param_grid, 
    scoring=scoring_metrics,
    cv=cv, refit="AUC", 
    n_jobs=-2, 
    verbose=3
)

In [22]:
print("Starting Grid Search for Model f1 (Treatment)...")
grid_1.fit(X_train_1, y_train_1)
best_model_1 = grid_1.best_estimator_
print(f"Model f1 Best CV ROC AUC: {np.round(grid_1.best_score_, 2)}")

Starting Grid Search for Model f1 (Treatment)...
Fitting 5 folds for each of 9 candidates, totalling 45 fits
[CV 1/5] END max_depth=10, n_estimators=50; AUC: (test=0.736) F1: (test=0.705) P: (test=0.672) R: (test=0.742) total time=   1.1s
[CV 2/5] END max_depth=10, n_estimators=50; AUC: (test=0.717) F1: (test=0.680) P: (test=0.685) R: (test=0.676) total time=   0.6s
[CV 3/5] END max_depth=10, n_estimators=50; AUC: (test=0.718) F1: (test=0.673) P: (test=0.659) R: (test=0.688) total time=   0.3s
[CV 4/5] END max_depth=10, n_estimators=50; AUC: (test=0.684) F1: (test=0.669) P: (test=0.638) R: (test=0.703) total time=   0.3s
[CV 5/5] END max_depth=10, n_estimators=50; AUC: (test=0.718) F1: (test=0.678) P: (test=0.647) R: (test=0.712) total time=   0.3s
[CV 1/5] END max_depth=10, n_estimators=150; AUC: (test=0.735) F1: (test=0.712) P: (test=0.675) R: (test=0.753) total time=   1.2s
[CV 2/5] END max_depth=10, n_estimators=150; AUC: (test=0.725) F1: (test=0.691) P: (test=0.688) R: (test=0.695

### **Counterfactuals (ATE): T-Learner Approach**

**Simulate A/B Counterfactuals on Test Set**

In [23]:
X_test_features = X_test[features]

# World A: Predict churn probability if EVERYONE got the discount (using Model f1)
prob_y1 = best_model_1.predict_proba(X_test_features)[:, 1]

# World B: Predict churn probability if NO ONE got the discount (using Model f0)
prob_y0 = best_model_0.predict_proba(X_test_features)[:, 1]

# Calculate Average Treatment Effect (ATE)
# This is like from s-learner: prob_y1 - prob_y2;
# where prob_y1 is the predicted churn probability if treated with discount, 
# and prob_y0 is the predicted churn probability if not treated.
ate = np.round(np.mean(prob_y1 - prob_y0), 3)
print(f"Average Treatment Effect (ATE) of Discount on Churn Probability (T-Learner):\n{ate}")

Average Treatment Effect (ATE) of Discount on Churn Probability (T-Learner):
-0.069


### **Inspection of Prediction Groups**

**Inspection of Prediction group A**

In [26]:
# Inspection of Counterfactual Group A (Trained with Treatment = 1) Gave discount
test_results_f1 = X_test.copy()
test_results_f1['predicted_churn_prob_if_treated'] = best_model_1.predict(X_test_features)
test_results_f1['actual_churn'] = y_test
print("--- Counterfactual Group A Preview ---")
test_results_f1.head()

--- Counterfactual Group A Preview ---


,tenure_months,weekly_logins,support_tickets,monthly_spend,discount_given,predicted_churn_prob_if_treated,actual_churn
9102,35,6,1,25,0,0,0
7868,5,2,2,50,0,1,1
4176,5,2,0,100,0,0,0
4161,4,2,1,10,1,1,1
8770,6,4,1,25,0,0,1


**Inspection of Prediction group B**

In [27]:
# Inspection of Counterfactual Group B (Trained with Treatment = 0) # No discount
test_results_f0 = X_test.copy()
test_results_f0['predicted_churn_prob_if_control'] = best_model_0.predict(X_test_features)
test_results_f0['actual_churn'] = y_test
print("--- Counterfactual Group B Preview ---")
test_results_f0.head()

--- Counterfactual Group B Preview ---


,tenure_months,weekly_logins,support_tickets,monthly_spend,discount_given,predicted_churn_prob_if_control,actual_churn
9102,35,6,1,25,0,0,0
7868,5,2,2,50,0,1,1
4176,5,2,0,100,0,0,0
4161,4,2,1,10,1,1,1
8770,6,4,1,25,0,1,1


### **Interpretation of ATE**

Because the T-learner isolates the treatment groups into two entirely separate models ($f_0$ and $f_1$), this **-6.9%** absolute drop is completely immune to the S-learner's risk of a tree algorithm accidentally ignoring the discount column.

Since estimated Average Treatment Effect (ATE) is **-0.069** (or roughly **-6.9%**), it means:

**What This Means for Your Business:**

1. **Stronger Causal Proof:** We can confidently tell stakeholders: *"When we give eligible high-risk customers a $20 discount, it causes an absolute **6.9 percentage point decrease** in their probability of churning."*
2. **Real-World Translation:** If a specific customer segment has a baseline predicted churn rate of **30%**, giving them the discount reduces their expected churn risk to **23.1%** ($30\% - 6.9\%$).
3. **Model Validation:** The fact that both the S-learner (-5.8%) and the T-learner (-6.9%) land in the same neighborhood tells us the causal signal is robust, stable, and not dependent on how the model was structured.